# Grad-CAM and gradient-based weights

**Weeks:** 25 · **Core question:** How do prediction gradients decide which channels matter?

**Prerequisites:** complete preceding notebooks in the curriculum. Each lesson targets 45–90 minutes including exercises.

## Concept and motivation

Grad-CAM weights a channel by the spatial average of the derivative of a class score with respect to that channel's activation. A useful sanity fact: for the specific *linear head after global average pooling* in our toy CNN, CAM and Grad-CAM weights are proportional, so their normalized maps should match. They differ in more general architectures.

**Key vocabulary:** gradient, backpropagation, target class logit, spatially pooled gradient.

**Learning goal:** Explain what a class-score gradient measures and when it differs from reference-aware comparison.

> All images and patient-like examples in this repository are synthetic teaching artifacts, not clinical data.


## Prepare the environment
Run this cell first. All core examples are offline and reproducible.

In [ ]:
from pathlib import Path
import sys
# Works when launched in the repository, even from a nested notebook directory.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'diffcam_edu').exists()), None)
if root is None:
    raise RuntimeError('Open Jupyter from the extracted diffcam_curriculum folder, or set working directory to it.')
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)


## Guided experiment
Execute the code cells in order. Before running each cell, predict what it will show; afterward, explain what changed and why.

### Experiment 1 of 3

In [ ]:
from diffcam_edu import make_cell_images
from diffcam_edu.models import fit_cnn,feature_maps,gradcam
from diffcam_edu.math import diffcam_map,normalize_map
x,y,_=make_cell_images(210,seed=44)
model,_=fit_cnn(x[:160],y[:160],epochs=7)
idx=180
sal,grad_weights=gradcam(model,x[idx],class_index=1)
A=feature_maps(model,x[idx])[0]
cam=diffcam_map(A,model.classifier.weight.detach().numpy()[1])
print('Grad-CAM weights first five:',grad_weights[:5])

### Experiment 2 of 3

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(7,3))
ax[0].imshow(normalize_map(cam),cmap='inferno');ax[0].set_title('CAM')
ax[1].imshow(normalize_map(sal),cmap='inferno');ax[1].set_title('Grad-CAM')
for a in ax:a.axis('off')
plt.show()
print('Maximum normalized difference:',np.max(np.abs(normalize_map(cam)-normalize_map(sal))))

### Experiment 3 of 3

In [ ]:
print('Weight scale factor: ',grad_weights[0]/(model.classifier.weight.detach().numpy()[1,0]+1e-12))
assert np.allclose(normalize_map(cam),normalize_map(sal),atol=1e-4)

## Exercises — complete independently
1. Explain *mathematically* why pooled class gradients equal classifier coefficients divided by map area here.
2. Name a change in classifier architecture that would break this equivalence.

## Check your understanding

**Exit ticket:** Explain what a class-score gradient measures and when it differs from reference-aware comparison.

Write a short answer and save a screenshot or plot from your experiment.

## Further reading
- [DiffCAM (CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Li_DiffCAM_Data-Driven_Saliency_Maps_by_Capturing_Feature_Differences_CVPR_2025_paper.html)
- [PyTorch tutorial hub](https://docs.pytorch.org/tutorials/)
- [Course roadmap](../COURSE_MAP.md)